In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 43.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 106.1 MB/s eta 0:00:0000:0100:01


In [2]:
# ============================================================
# 0) Imports
# ============================================================
import os, random, math
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# PennyLane (for quantum transform)
import pennylane as qml

# ============================================================
# 1) Reproducibility helpers
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 2) MVTec-AD Hazelnut Dataset Loader (NO MNIST)
# ============================================================
class MVTecHazelnutDataset(Dataset):
    """
    Assumes MVTec structure like:
    root/
      Hazelnut/
        train/
          good/xxx.png
        test/
          good/xxx.png
          broken_large/xxx.png
          broken_small/xxx.png
          contamination/xxx.png
        ground_truth/...
    For AE anomaly: train on train/good only.
    For test: label good=0, anomaly=1.
    """
    def __init__(self, root, split="train", image_size=256):
        super().__init__()
        self.root = root
        self.split = split
        self.image_size = image_size

        base = os.path.join(root, "hazelnut", split)
        self.samples = []

        if split == "train":
            good_dir = os.path.join(base, "good")
            for fn in sorted(os.listdir(good_dir)):
                if fn.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                    self.samples.append((os.path.join(good_dir, fn), 0))
        else:
            # test: multiple folders
            for cls in sorted(os.listdir(base)):
                cls_dir = os.path.join(base, cls)
                if not os.path.isdir(cls_dir):
                    continue
                label = 0 if cls == "good" else 1
                for fn in sorted(os.listdir(cls_dir)):
                    if fn.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                        self.samples.append((os.path.join(cls_dir, fn), label))

    def __len__(self):
        return len(self.samples)

    def _load_img(self, path):
        img = Image.open(path).convert("RGB")
        # resize
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        arr = np.asarray(img).astype(np.float32) / 255.0  # [0,1]
        # CHW
        arr = np.transpose(arr, (2, 0, 1))
        return torch.from_numpy(arr)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        x = self._load_img(path)
        return x, torch.tensor(y, dtype=torch.long)


def get_loaders_mvtec(root, image_size=256, train_frac=1.0, seed=1, batch_size=32):
    """
    Returns:
      train_loader (train/good subset by train_frac)
      test_loader  (full test)
    """
    set_seed(seed)
    train_ds = MVTecHazelnutDataset(root=root, split="train", image_size=image_size)
    test_ds  = MVTecHazelnutDataset(root=root, split="test",  image_size=image_size)

    # Subsample train (only good exists in train split)
    n = len(train_ds)
    idxs = np.arange(n)
    np.random.shuffle(idxs)
    k = max(1, int(train_frac * n))
    idxs = idxs[:k]
    train_sub = Subset(train_ds, idxs.tolist())

    train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


# ============================================================
# 3) Models: Classical AE + Ablation MLP-AE
# ============================================================
class ConvAutoencoder(nn.Module):
    """
    Simple conv AE for 256x256x3 (works for other sizes too)
    latent_dim=64 default as you used.
    """
    def __init__(self, latent_dim=64):
        super().__init__()
        # Encoder
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),  # 256->128
            nn.ReLU(True),
            nn.Conv2d(32, 64, 4, 2, 1), # 128->64
            nn.ReLU(True),
            nn.Conv2d(64, 128, 4, 2, 1),# 64->32
            nn.ReLU(True),
            nn.Conv2d(128, 256, 4, 2, 1),# 32->16
            nn.ReLU(True),
        )
        self.enc_fc = nn.Linear(256*16*16, latent_dim)

        # Decoder
        self.dec_fc = nn.Linear(latent_dim, 256*16*16)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), # 16->32
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 32->64
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),   # 64->128
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),    # 128->256
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.flatten(1)
        z = self.enc_fc(h)
        return z

    def decode(self, z):
        h = self.dec_fc(z)
        h = h.view(z.shape[0], 256, 16, 16)
        xhat = self.dec(h)
        return xhat

    def forward(self, x):
        return self.decode(self.encode(x))


class ClassicalLatentMLP(nn.Module):
    def __init__(self, latent_dim=64, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, z):
        return self.net(z)


class AblationMLPAE(nn.Module):
    def __init__(self, latent_dim=64, hidden_dim=32):
        super().__init__()
        self.ae = ConvAutoencoder(latent_dim=latent_dim)
        self.mlp = ClassicalLatentMLP(latent_dim=latent_dim, hidden_dim=hidden_dim)

    def forward(self, x):
        z = self.ae.encode(x)
        z2 = self.mlp(z)
        return self.ae.decode(z2)


# ============================================================
# 4) Hybrid Quantum AE (robust version: quantum forward, no Q-grad)
# ============================================================
class QuantumLatentLayerNoGrad(nn.Module):
    """
    Robust quantum latent transform:
    - runs on CPU
    - converts z -> cpu numpy (DETACHED) so PennyLane doesn't try autograd on Torch
    - returns torch tensor back on original device
    This avoids ALL errors you saw, but quantum parameters are NOT trained by backprop.
    """
    def __init__(self, latent_dim=64, n_qubits=4, n_layers=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # simple classical projections around quantum block
        self.in_proj  = nn.Linear(latent_dim, n_qubits)
        self.out_proj = nn.Linear(n_qubits, latent_dim)

        # quantum params (can be trained manually later; here kept as torch params but not autograd-connected)
        self.q_weights = nn.Parameter(0.01 * torch.randn(n_layers, n_qubits, 3))

        self.dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(self.dev, interface="numpy")  # IMPORTANT: no torch interface
        def circuit(x, w):
            # x: (n_qubits,) numpy
            for i in range(n_qubits):
                qml.RY(x[i], wires=i)
            for l in range(n_layers):
                for i in range(n_qubits):
                    qml.Rot(w[l, i, 0], w[l, i, 1], w[l, i, 2], wires=i)
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i+1])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self._circuit = circuit

    def forward(self, z):
        # z: (B, latent_dim) on cuda or cpu
        device = z.device
        dtype = z.dtype

        x = torch.tanh(self.in_proj(z))  # (B, n_qubits) stays differentiable up to here

        # detach before sending to numpy, to avoid "requires grad -> numpy" error
        x_cpu = x.detach().cpu().numpy()  # (B, n_qubits)
        w_cpu = self.q_weights.detach().cpu().numpy()

        outs = []
        for i in range(x_cpu.shape[0]):
            yi = np.array(self._circuit(x_cpu[i], w_cpu), dtype=np.float32)  # (n_qubits,)
            outs.append(yi)
        q = torch.tensor(np.stack(outs, axis=0), dtype=dtype, device=device)  # (B, n_qubits)

        q = torch.tanh(q)
        zq = self.out_proj(q)  # back to latent_dim (trainable classical)
        return zq


class HybridQuantumAE(nn.Module):
    def __init__(self, latent_dim=64, n_qubits=4, n_layers=2):
        super().__init__()
        self.ae = ConvAutoencoder(latent_dim=latent_dim)
        self.q  = QuantumLatentLayerNoGrad(latent_dim=latent_dim, n_qubits=n_qubits, n_layers=n_layers)

    def forward(self, x):
        z  = self.ae.encode(x)
        zq = self.q(z)
        return self.ae.decode(zq)


# ============================================================
# 5) Training + Scoring + Metrics
# ============================================================
def train_ae(model, train_loader, device, epochs=50, lr=1e-3):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs+1):
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)
        running = 0.0
        for x, _ in pbar:
            x = x.to(device)
            opt.zero_grad(set_to_none=True)
            x_hat = model(x)
            loss = F.mse_loss(x_hat, x)
            loss.backward()
            opt.step()
            running += loss.item()
            pbar.set_postfix(loss=running / max(1, (pbar.n+1)))


@torch.no_grad()
def recon_errors(model, loader, device):
    model.eval()
    scores = []
    labels = []
    for x, y in loader:
        x = x.to(device)
        x_hat = model(x)
        # per-sample MSE
        err = F.mse_loss(x_hat, x, reduction="none")
        err = err.flatten(1).mean(1)
        scores.append(err.detach().cpu())
        labels.append(y.detach().cpu())
    scores = torch.cat(scores).numpy()
    labels = torch.cat(labels).numpy()
    return scores, labels


def best_f1_threshold(scores, labels, n_steps=200):
    # sweep thresholds between min and max
    lo, hi = float(scores.min()), float(scores.max())
    best = (-1, None)
    for t in np.linspace(lo, hi, n_steps):
        preds = (scores >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best[0]:
            best = (f1, t)
    return best[0], best[1]


def compute_metrics(scores, labels):
    auroc = roc_auc_score(labels, scores)
    auprc = average_precision_score(labels, scores)
    f1, thr = best_f1_threshold(scores, labels)
    preds = (scores >= thr).astype(int)

    TP = int(((preds==1) & (labels==1)).sum())
    TN = int(((preds==0) & (labels==0)).sum())
    FP = int(((preds==1) & (labels==0)).sum())
    FN = int(((preds==0) & (labels==1)).sum())
    return {"AUROC": auroc, "AUPRC": auprc, "F1": f1, "thr": float(thr), "TP": TP, "TN": TN, "FP": FP, "FN": FN}


# ============================================================
# 6) Main experiment runner (same style as your Classical AE table)
# ============================================================
def run_experiments(
    mvtec_root,
    image_size=256,
    latent_dim=64,
    batch_size=8,
    epochs=50,
    lr=1e-3,
    train_fracs=(0.10, 0.25, 0.50, 1.00),
    seeds=(1,2,3),
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    all_rows = []

    for train_frac in train_fracs:
        for seed in seeds:
            set_seed(seed)

            train_loader, test_loader = get_loaders_mvtec(
                root=mvtec_root,
                image_size=image_size,
                train_frac=train_frac,
                seed=seed,
                batch_size=batch_size
            )

            # -------- Classical AE --------
            m_c = ConvAutoencoder(latent_dim=latent_dim).to(device)
            train_ae(m_c, train_loader, device, epochs=epochs, lr=lr)
            scores_c, labels = recon_errors(m_c, test_loader, device)
            met_c = compute_metrics(scores_c, labels)
            all_rows.append({"model":"Classical_AE", **met_c, "train_frac":train_frac, "seed":seed})

            # -------- Ablation MLP AE --------
            m_a = AblationMLPAE(latent_dim=latent_dim, hidden_dim=32).to(device)
            train_ae(m_a, train_loader, device, epochs=epochs, lr=lr)
            scores_a, _ = recon_errors(m_a, test_loader, device)
            met_a = compute_metrics(scores_a, labels)
            all_rows.append({"model":"Ablation_MLP_AE", **met_a, "train_frac":train_frac, "seed":seed})

            # -------- Hybrid Quantum AE (robust) --------
            m_q = HybridQuantumAE(latent_dim=latent_dim, n_qubits=4, n_layers=2).to(device)
            train_ae(m_q, train_loader, device, epochs=epochs, lr=lr)
            scores_q, _ = recon_errors(m_q, test_loader, device)
            met_q = compute_metrics(scores_q, labels)
            all_rows.append({"model":"Hybrid_Quantum_AE", **met_q, "train_frac":train_frac, "seed":seed})

            print(f"[done] frac={train_frac} seed={seed}")

    df = pd.DataFrame(all_rows)
    return df


def summarize(df):
    # mean ± std per train_frac and model
    summ = (df.groupby(["model","train_frac"])
              .agg({"AUROC":["mean","std"], "AUPRC":["mean","std"], "F1":["mean","std"]}))
    return summ


# ============================================================
# 7) RUN (edit mvtec_root path!)
# ============================================================
# Example:
# mvtec_root = "/path/to/mvtec_ad"
# df_all = run_experiments(mvtec_root, image_size=256, batch_size=8, epochs=50, lr=1e-3)
# print("Raw results:\n", df_all)
# print("\nSummary (mean ± std):\n", summarize(df_all))
# out_xlsx = "mvtec_hazelnut_3way_results.xlsx"
# df_all.to_excel(out_xlsx, index=False)
# print("Saved:", out_xlsx)


In [3]:
mvtec_root = "/kaggle/input/mvtec-ad/"   # or wherever hazelnut folder exists

In [4]:
df_all = run_experiments(
    mvtec_root,
    image_size=256,     # classical AE used 256x256 based on your conv AE design
    batch_size=8,
    epochs=50,          # ✅ paper-worthy
    lr=1e-3
)
print("Raw results:\n", df_all)
print("\nSummary (mean ± std):\n", summarize(df_all))

df_all.to_excel("mvtec_hazelnut_3way_results.xlsx", index=False)
print("Saved: mvtec_hazelnut_3way_results.xlsx")


Epoch 1/50:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 5/50:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 47/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/49 [00:00<?, ?it/s]

[done] frac=1.0 seed=3
Raw results:
                 model     AUROC     AUPRC        F1       thr  TP  TN  FP  FN  \
0        Classical_AE  0.771786  0.867938  0.810127  0.001213  64  16  24   6   
1     Ablation_MLP_AE  0.783929  0.871220  0.827160  0.001135  67  15  25   3   
2   Hybrid_Quantum_AE  0.732500  0.847935  0.797546  0.001186  65  12  28   5   
3        Classical_AE  0.789286  0.874476  0.833333  0.001316  65  19  21   5   
4     Ablation_MLP_AE  0.758214  0.862847  0.809816  0.001285  66  13  27   4   
5   Hybrid_Quantum_AE  0.745000  0.852023  0.807947  0.001327  61  20  20   9   
6        Classical_AE  0.735714  0.857613  0.779661  0.001167  69   2  38   1   
7     Ablation_MLP_AE  0.792500  0.882034  0.819876  0.001075  66  15  25   4   
8   Hybrid_Quantum_AE  0.683214  0.823040  0.777778  0.001242  70   0  40   0   
9        Classical_AE  0.827143  0.905222  0.818792  0.001086  61  22  18   9   
10    Ablation_MLP_AE  0.778214  0.875110  0.805195  0.001284  62  18  2

In [5]:
# ============================================================
# 8) Accuracy from best-F1 threshold + show best run per model
# ============================================================
def add_accuracy_rowwise(df):
    df = df.copy()
    df["accuracy"] = (df["TP"] + df["TN"]) / (df["TP"] + df["TN"] + df["FP"] + df["FN"])
    return df

df_all2 = add_accuracy_rowwise(df_all)

# pick best AUROC run per model (you can change to best F1 if you want)
best_runs = df_all2.sort_values(["model", "AUROC"], ascending=[True, False]).groupby("model").head(1)
print("Best run per model:\n", best_runs[["model","train_frac","seed","AUROC","AUPRC","F1","accuracy","thr","TP","TN","FP","FN"]])

# summary including accuracy
summary_with_acc = (df_all2.groupby(["model","train_frac"])
                    .agg(AUROC_mean=("AUROC","mean"), AUROC_std=("AUROC","std"),
                         AUPRC_mean=("AUPRC","mean"), AUPRC_std=("AUPRC","std"),
                         F1_mean=("F1","mean"), F1_std=("F1","std"),
                         ACC_mean=("accuracy","mean"), ACC_std=("accuracy","std")))
print("\nSummary (mean ± std) incl. accuracy:\n", summary_with_acc)


Best run per model:
                 model  train_frac  seed     AUROC     AUPRC        F1  \
19    Ablation_MLP_AE         0.5     1  0.892500  0.937134  0.875912   
30       Classical_AE         1.0     2  0.919286  0.955215  0.890411   
23  Hybrid_Quantum_AE         0.5     2  0.754643  0.854948  0.818182   

    accuracy       thr  TP  TN  FP  FN  
19  0.845455  0.000956  60  33   7  10  
30  0.854545  0.000714  65  29  11   5  
23  0.745455  0.001409  63  19  21   7  

Summary (mean ± std) incl. accuracy:
                               AUROC_mean  AUROC_std  AUPRC_mean  AUPRC_std  \
model             train_frac                                                 
Ablation_MLP_AE   0.10          0.778214   0.017843    0.872034   0.009619   
                  0.25          0.776310   0.020424    0.873958   0.012499   
                  0.50          0.870476   0.019081    0.925905   0.009792   
                  1.00          0.861429   0.019444    0.924042   0.013645   
Classical_AE   

In [ ]:
# ============================================================
# 9) Train final models for visualization & saving
#     (use train_frac=1.0, seed=1 by default)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

FINAL_SEED = 1
FINAL_TRAIN_FRAC = 1.0

set_seed(FINAL_SEED)

train_loader_vis, test_loader_vis = get_loaders_mvtec(
    root=mvtec_root,
    image_size=256,
    train_frac=FINAL_TRAIN_FRAC,
    seed=FINAL_SEED,
    batch_size=8
)

# ---- Classical AE ----
model_c = ConvAutoencoder(latent_dim=64).to(device)
train_ae(model_c, train_loader_vis, device, epochs=50, lr=1e-3)

# ---- Ablation MLP AE ----
model_a = AblationMLPAE(latent_dim=64, hidden_dim=32).to(device)
train_ae(model_a, train_loader_vis, device, epochs=50, lr=1e-3)

# ---- Hybrid Quantum AE ----
model_q = HybridQuantumAE(latent_dim=64, n_qubits=4, n_layers=2).to(device)
train_ae(model_q, train_loader_vis, device, epochs=50, lr=1e-3)

print("Final models trained ✅")


Using device: cuda


Epoch 1/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/49 [00:00<?, ?it/s]

...

Epoch 8/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 9/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 10/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 11/50:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 12/50:   0%|          | 0/49 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# 10) Final metrics + anomaly scores
# ============================================================
def evaluate_and_print(model, loader, device, name="model"):
    scores, labels = recon_errors(model, loader, device)
    metrics = compute_metrics(scores, labels)
    acc = (metrics["TP"] + metrics["TN"]) / (metrics["TP"] + metrics["TN"] + metrics["FP"] + metrics["FN"])
    metrics["accuracy"] = acc
    print(f"\n{name} metrics:", metrics)
    return scores, labels, metrics

scores_c, labels, met_c = evaluate_and_print(model_c, test_loader_vis, device, "Classical_AE")
scores_a, _,      met_a = evaluate_and_print(model_a, test_loader_vis, device, "Ablation_MLP_AE")
scores_q, _,      met_q = evaluate_and_print(model_q, test_loader_vis, device, "Hybrid_Quantum_AE")


In [ ]:
# ============================================================
# 11) Anomaly-score plots
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

def plot_score_hist(scores, labels, title):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    plt.figure(figsize=(6,4))
    plt.hist(scores[labels==0], bins=40, alpha=0.6, label="good (0)")
    plt.hist(scores[labels==1], bins=40, alpha=0.6, label="anomaly (1)")
    plt.title(title)
    plt.xlabel("Reconstruction error (anomaly score)")
    plt.ylabel("Count")
    plt.legend()
    plt.show()

def plot_roc(scores, labels, title):
    fpr, tpr, _ = roc_curve(labels, scores)
    plt.figure(figsize=(5,5))
    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1], linestyle="--")
    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.show()

plot_score_hist(scores_c, labels, "Classical AE score distribution")
plot_score_hist(scores_a, labels, "Ablation MLP AE score distribution")
plot_score_hist(scores_q, labels, "Hybrid Quantum AE score distribution")

plot_roc(scores_c, labels, "Classical AE ROC")
plot_roc(scores_a, labels, "Ablation MLP AE ROC")
plot_roc(scores_q, labels, "Hybrid Quantum AE ROC")


In [ ]:
# ============================================================
# 12) Reconstruction visualization
# ============================================================
@torch.no_grad()
def visualize_reconstructions(model, loader, device, n=5, title="Model"):
    model.eval()

    # collect a batch
    x, y = next(iter(loader))
    x = x.to(device)
    x_hat = model(x)

    x = x.cpu().numpy()
    x_hat = x_hat.cpu().numpy()
    y = y.numpy()

    # pick some good and some anomaly indices if available
    good_idx = np.where(y==0)[0][:n]
    anom_idx = np.where(y==1)[0][:n]

    def show_rows(idxs, label_name):
        if len(idxs) == 0:
            print(f"No {label_name} samples found in the first batch. Try running again.")
            return

        plt.figure(figsize=(10, 2*len(idxs)))
        for row, i in enumerate(idxs):
            # original
            plt.subplot(len(idxs), 2, 2*row+1)
            img = np.transpose(x[i], (1,2,0))
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"{title} - {label_name} ORIG")

            # recon
            plt.subplot(len(idxs), 2, 2*row+2)
            img2 = np.transpose(x_hat[i], (1,2,0))
            plt.imshow(img2)
            plt.axis("off")
            plt.title(f"{title} - {label_name} RECON")
        plt.tight_layout()
        plt.show()

    show_rows(good_idx, "GOOD")
    show_rows(anom_idx, "ANOM")

visualize_reconstructions(model_c, test_loader_vis, device, n=5, title="Classical AE")
visualize_reconstructions(model_a, test_loader_vis, device, n=5, title="Ablation MLP AE")
visualize_reconstructions(model_q, test_loader_vis, device, n=5, title="Hybrid Quantum AE")


In [ ]:
# ============================================================
# 13) Save models + final excel
# ============================================================
torch.save(model_c.state_dict(), "classical_ae_best.pt")
torch.save(model_a.state_dict(), "ablation_mlp_ae_best.pt")
torch.save(model_q.state_dict(), "hybrid_quantum_ae_best.pt")
print("Saved model checkpoints ✅")

final_compare = pd.DataFrame([
    {"model":"Classical_AE", **met_c},
    {"model":"Ablation_MLP_AE", **met_a},
    {"model":"Hybrid_Quantum_AE", **met_q},
])

final_compare["accuracy"] = (final_compare["TP"] + final_compare["TN"]) / (final_compare["TP"] + final_compare["TN"] + final_compare["FP"] + final_compare["FN"])
final_compare.to_excel("mvtec_hazelnut_final_compare.xlsx", index=False)
print("Saved: mvtec_hazelnut_final_compare.xlsx")


In [ ]:
# ============================================================
# 14) Pixel-level anomaly maps + overlay visualization
# ============================================================
import matplotlib.pyplot as plt

@torch.no_grad()
def visualize_anomaly_maps(model, loader, device, n=5, title="Model"):
    model.eval()

    # get one batch
    batch = next(iter(loader))
    if len(batch) == 3:
        x, y, paths = batch
    else:
        x, y = batch
        paths = [None]*len(y)

    x = x.to(device)
    x_hat = model(x)

    x_np = x.cpu().numpy()
    xhat_np = x_hat.cpu().numpy()
    y_np = y.cpu().numpy()

    # pixel-wise absolute error (mean over channels)
    # shape: (B, H, W)
    anomaly_maps = np.mean(np.abs(x_np - xhat_np), axis=1)

    # pick good + anomaly
    good_idx = np.where(y_np == 0)[0][:n]
    anom_idx = np.where(y_np == 1)[0][:n]

    def show(idxs, label_name):
        if len(idxs) == 0:
            print(f"No {label_name} samples in this batch.")
            return

        for i in idxs:
            orig = np.transpose(x_np[i], (1,2,0))
            recon = np.transpose(xhat_np[i], (1,2,0))
            amap = anomaly_maps[i]

            plt.figure(figsize=(12,4))

            # Original
            plt.subplot(1,4,1)
            plt.imshow(orig)
            plt.title(f"{title} - {label_name}\nOriginal")
            plt.axis("off")

            # Reconstruction
            plt.subplot(1,4,2)
            plt.imshow(recon)
            plt.title("Reconstruction")
            plt.axis("off")

            # Anomaly map (heatmap)
            plt.subplot(1,4,3)
            plt.imshow(amap, cmap="hot")
            plt.colorbar(fraction=0.046, pad=0.04)
            plt.title("Anomaly Map |x - x̂|")
            plt.axis("off")

            # Overlay
            plt.subplot(1,4,4)
            plt.imshow(orig)
            plt.imshow(amap, cmap="jet", alpha=0.5)
            plt.title("Overlay")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

    print(f"\n=== {title}: GOOD samples ===")
    show(good_idx, "GOOD")

    print(f"\n=== {title}: ANOMALY samples ===")
    show(anom_idx, "ANOM")


# Run for all three models
visualize_anomaly_maps(model_c, test_loader_vis, device, n=3, title="Classical AE")
visualize_anomaly_maps(model_a, test_loader_vis, device, n=3, title="Ablation MLP AE")
visualize_anomaly_maps(model_q, test_loader_vis, device, n=3, title="Hybrid Quantum AE")


In [ ]:
# ============================================================
# 15) Single-image anomaly inspection by file path
# ============================================================
import matplotlib.pyplot as plt

def load_and_preprocess_image(img_path, image_size=256):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((image_size, image_size), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))  # CHW
    x = torch.from_numpy(arr).unsqueeze(0)  # (1,3,H,W)
    return x


@torch.no_grad()
def inspect_image_by_path(model, img_path, device, image_size=256, title="Model"):
    model.eval()

    # load
    x = load_and_preprocess_image(img_path, image_size=image_size).to(device)

    # forward
    x_hat = model(x)

    x_np = x.cpu().numpy()[0]
    xhat_np = x_hat.cpu().numpy()[0]

    # scalar anomaly score
    scalar_score = float(F.mse_loss(x_hat, x).item())

    # pixel-level anomaly map
    anomaly_map = np.mean(np.abs(x_np - xhat_np), axis=0)  # (H,W)

    orig = np.transpose(x_np, (1,2,0))
    recon = np.transpose(xhat_np, (1,2,0))

    plt.figure(figsize=(12,4))

    # Original
    plt.subplot(1,4,1)
    plt.imshow(orig)
    plt.title("Original")
    plt.axis("off")

    # Reconstruction
    plt.subplot(1,4,2)
    plt.imshow(recon)
    plt.title("Reconstruction")
    plt.axis("off")

    # Anomaly Map
    plt.subplot(1,4,3)
    plt.imshow(anomaly_map, cmap="hot")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title("Anomaly Map |x - x̂|")
    plt.axis("off")

    # Overlay
    plt.subplot(1,4,4)
    plt.imshow(orig)
    plt.imshow(anomaly_map, cmap="jet", alpha=0.5)
    plt.title("Overlay")
    plt.axis("off")

    plt.suptitle(f"{title} | Anomaly score = {scalar_score:.6f}", fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"[{title}] Scalar anomaly score:", scalar_score)
    return scalar_score


In [ ]:
img_path = "/kaggle/input/mvtec-ad/hazelnut/test/good/005.png"
inspect_image_by_path(model_c, img_path, device, title="Classical AE")

In [ ]:
img_path = "/kaggle/input/mvtec-ad/hazelnut/test/hole/002.png"
inspect_image_by_path(model_c, img_path, device, title="Classical AE")


In [ ]:
img_path = "/kaggle/input/mvtec-ad/hazelnut/test/crack/003.png"

inspect_image_by_path(model_c, img_path, device, title="Classical AE")
inspect_image_by_path(model_a, img_path, device, title="Ablation MLP AE")
inspect_image_by_path(model_q, img_path, device, title="Hybrid Quantum AE")
